In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, f1_score

In [2]:
data = pd.read_csv('/kaggle/input/datasets/fedesoriano/stroke-prediction-dataset/healthcare-dataset-stroke-data.csv')
data

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1
...,...,...,...,...,...,...,...,...,...,...,...,...
5105,18234,Female,80.0,1,0,Yes,Private,Urban,83.75,NaN,never smoked,0
5106,44873,Female,81.0,0,0,Yes,Self-employed,Urban,125.20,40.0,never smoked,0
5107,19723,Female,35.0,0,0,Yes,Self-employed,Rural,82.99,30.6,never smoked,0
5108,37544,Male,51.0,0,0,Yes,Private,Rural,166.29,25.6,formerly smoked,0


In [3]:
data.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [4]:
data = data.drop(columns='id')

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   gender             5110 non-null   object 
 1   age                5110 non-null   float64
 2   hypertension       5110 non-null   int64  
 3   heart_disease      5110 non-null   int64  
 4   ever_married       5110 non-null   object 
 5   work_type          5110 non-null   object 
 6   Residence_type     5110 non-null   object 
 7   avg_glucose_level  5110 non-null   float64
 8   bmi                4909 non-null   float64
 9   smoking_status     5110 non-null   object 
 10  stroke             5110 non-null   int64  
dtypes: float64(3), int64(3), object(5)
memory usage: 439.3+ KB


In [6]:
data.shape

(5110, 11)

In [7]:
data.describe()

,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
count,5110.000000,5110.000000,5110.000000,5110.000000,4909.000000,5110.000000
mean,43.226614,0.097456,0.054012,106.147677,28.893237,0.048728
std,22.612647,0.296607,0.226063,45.283560,7.854067,0.215320
min,0.080000,0.000000,0.000000,55.120000,10.300000,0.000000
25%,25.000000,0.000000,0.000000,77.245000,23.500000,0.000000
50%,45.000000,0.000000,0.000000,91.885000,28.100000,0.000000
75%,61.000000,0.000000,0.000000,114.090000,33.100000,0.000000
max,82.000000,1.000000,1.000000,271.740000,97.600000,1.000000


In [8]:
data.isna().sum()

gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64

In [9]:
data.duplicated().sum()

np.int64(0)

In [10]:
data = data.dropna(subset=['bmi']).reset_index(drop=True)

In [11]:
x = data.drop(columns='stroke')
y = data['stroke']

In [12]:
X_train, X_temp, y_train, y_temp = train_test_split(
    x,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)


In [13]:
categorical = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']  # FIX: spelling
numerical = ['age', 'avg_glucose_level', 'bmi']

for i in categorical:
    print(X_train[i].value_counts(normalize=True)*100)
    print('_'*10)

gender
Female    59.458673
Male      40.541327
Name: proportion, dtype: float64
__________
ever_married
Yes    66.268917
No     33.731083
Name: proportion, dtype: float64
__________
work_type
Private          57.566938
Self-employed    16.094296
Govt_job         13.154831
children         12.805588
Never_worked      0.378347
Name: proportion, dtype: float64
__________
Residence_type
Urban    50.523865
Rural    49.476135
Name: proportion, dtype: float64
__________
smoking_status
never smoked       37.892899
Unknown            29.103609
formerly smoked    17.491269
smokes             15.512224
Name: proportion, dtype: float64
__________


In [14]:
outliers = ['avg_glucose_level', 'bmi']

for j in outliers:
    Q1 = X_train[j].quantile(0.25)
    Q3 = X_train[j].quantile(0.75)
    IQR = Q3-Q1

    lower = Q1 - 1.5 * IQR
    upper =  Q3 + 1.5 * IQR

    print(f'{j} : the lower value is {lower}')
    print(f'{j} : the upper value is {upper}')

avg_glucose_level : the lower value is 21.37749999999999
avg_glucose_level : the upper value is 170.2975
bmi : the lower value is 9.350000000000001
bmi : the upper value is 47.35


In [15]:
scaling_col = ['age', 'avg_glucose_level', 'bmi']
encoding_col = ['gender','ever_married', 'work_type', 'Residence_type', 'smoking_status']

categorical_pipeline = Pipeline([
    ('encoding', OneHotEncoder(handle_unknown='ignore'))
])

# bmi_pipeline = Pipeline([
#     ('imputer', SimpleImputer(strategy='mean')),
#     ('scaler', MinMaxScaler())
# ])
scaling_pipeline = Pipeline([
    ('scaling', MinMaxScaler())
])


# column Trasnforming
ct = ColumnTransformer([
    ('scaling',scaling_pipeline, scaling_col ),
    ('encoding',categorical_pipeline, encoding_col ),
], remainder='passthrough')

In [16]:
# fitting
X_train_encoded = ct.fit_transform(X_train)

X_val_encoded = ct.transform(X_val)
X_test_encoded = ct.transform(X_test)

print(X_train_encoded.shape, y_train.shape)
print(X_val_encoded.shape, y_val.shape)
print(X_test_encoded.shape, y_test.shape)
print(X_train_encoded.dtype)

(3436, 20) (3436,)
(736, 20) (736,)
(737, 20) (737,)
float64


In [17]:
import joblib

joblib.dump(ct, '/kaggle/working/preprocessor.joblib')

print("Preprocessor saved successfully")

Preprocessor saved successfully


In [18]:
classes = np.unique(y_train) # array([0, 1])

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)
class_weight = dict(zip(classes, weights))
class_weight

{np.int64(0): np.float64(0.5221884498480243),
 np.int64(1): np.float64(11.767123287671232)}

In [19]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import AUC, Recall, Precision
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

In [20]:
inputs = Input(shape=(X_train_encoded.shape[1],))

x1 = Dense(64, activation='relu')(inputs)
x2 = Dropout(0.3)(x1)

x3 = Dense(64, activation='relu')(x2)
x4 = Dropout(0.3)(x3)

output = Dense(1, activation='sigmoid')(x4)

model = Model(inputs, output)

model.summary()

I0000 00:00:1792414984.796997      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1792414984.800605      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         1,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,569 (21.75 KB)

 Trainable params: 5,569 (21.75 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    patience=5,
    factor=0.1
)

model_point = ModelCheckpoint('best.weights.h5', monitor='val_loss',
                              save_best_only=True, save_weights_only=True)

In [22]:
model.compile(
    loss=BinaryCrossentropy,
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy', Recall(name='recall'), Precision(name='precision'),
             AUC(curve='PR', name='pr_auc')]
)

In [23]:
history = model.fit(
    X_train_encoded,
    y_train,
    validation_data=(X_val_encoded,y_val),
    batch_size=128,
    epochs=100,
    class_weight=class_weight,
    callbacks=[early_stop, reduce_lr, model_point]
)

Epoch 1/100
21/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7242 - loss: 0.6972 - pr_auc: 0.0383 - precision: 0.0378 - recall: 0.2399 

I0000 00:00:1792414990.607676     141 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


27/27 ━━━━━━━━━━━━━━━━━━━━ 9s 201ms/step - accuracy: 0.6569 - loss: 0.6963 - pr_auc: 0.0521 - precision: 0.0489 - recall: 0.3836 - val_accuracy: 0.5000 - val_loss: 0.6673 - val_pr_auc: 0.0870 - val_precision: 0.0577 - val_recall: 0.7097 - learning_rate: 0.0010
Epoch 2/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5885 - loss: 0.6502 - pr_auc: 0.0681 - precision: 0.0699 - recall: 0.7055 - val_accuracy: 0.5530 - val_loss: 0.6230 - val_pr_auc: 0.0832 - val_precision: 0.0592 - val_recall: 0.6452 - learning_rate: 0.0010
Epoch 3/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6065 - loss: 0.6185 - pr_auc: 0.0953 - precision: 0.0717 - recall: 0.6918 - val_accuracy: 0.5448 - val_loss: 0.5944 - val_pr_auc: 0.0873 - val_precision: 0.0607 - val_recall: 0.6774 - learning_rate: 0.0010
Epoch 4/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6173 - loss: 0.5882 - pr_auc: 0.1244 - precision: 0.0768 - recall: 0.7260 - val_accuracy: 0.6576 - val_loss: 0.5393 - val_pr_auc: 0.

In [24]:
test_loss, test_accuracy, test_pr_auc, test_precision, test_recall = model.evaluate(
    X_test_encoded,
    y_test,
    verbose=1
)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)
print("Test PR-AUC:", test_pr_auc)
print("Test Precision:", test_precision)
print("Test Recall:", test_recall)

24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.7761 - loss: 0.4567 - pr_auc: 0.1852 - precision: 0.1200 - recall: 0.6562
Test Loss: 0.456711083650589
Test Accuracy: 0.7761194109916687
Test PR-AUC: 0.65625
Test Precision: 0.11999999731779099
Test Recall: 0.18520332872867584


In [25]:
print(model.metrics_names)
print(model.evaluate(X_test_encoded, y_test, return_dict=True, verbose=0))

['loss', 'compile_metrics']
{'accuracy': 0.7761194109916687, 'loss': 0.456711083650589, 'pr_auc': 0.18520332872867584, 'precision': 0.11999999731779099, 'recall': 0.65625}


In [26]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred_prob = model.predict(X_test_encoded).ravel()
y_pred = (y_pred_prob >= 0.5).astype(int)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
[[551 154]
 [ 11  21]]
              precision    recall  f1-score   support

           0       0.98      0.78      0.87       705
           1       0.12      0.66      0.20        32

    accuracy                           0.78       737
   macro avg       0.55      0.72      0.54       737
weighted avg       0.94      0.78      0.84       737

